# Lab 2 — Robótica Móvil con TurtleBot3, ROS2 y Gazebo

**Objetivos de la práctica:**
1. Comprender el modelo cinemático de un robot de tracción diferencial.
2. Conocer la plataforma TurtleBot3 Burger y sus componentes.
3. Familiarizarse con ROS2 y Gazebo como herramientas de desarrollo robótico.
4. Realizar el mapeo de un entorno simulado.
5. Implementar y ejecutar una trayectoria cerrada propia con curvas y rectas.

---

## 1. Modelo de Robot Diferencial

### 1.1 ¿Qué es un robot diferencial?

Un **robot de tracción diferencial** es el tipo de locomoción móvil más común en robótica terrestre. Está compuesto por **dos ruedas motrices independientes** colocadas sobre el mismo eje, más una o más ruedas de apoyo pasivas (castor) que solo sirven para balance.

El movimiento se controla ajustando de forma diferencial la velocidad de cada rueda:

| Condición | Resultado |
|-----------|----------|
| `v_izq = v_der` | Avanza/retrocede en línea recta |
| `v_izq > v_der` | Gira a la derecha |
| `v_izq < v_der` | Gira a la izquierda |
| `v_izq = -v_der` | Giro en el lugar (rotación pura) |

### 1.2 Modelo cinemático

Las dos velocidades que definen completamente el movimiento son:

- **Velocidad lineal** $v$ [m/s]: qué tan rápido avanza el centro del robot.
- **Velocidad angular** $\omega$ [rad/s]: qué tan rápido rota.

Relacionadas con las velocidades de cada rueda por:

$$v = \frac{v_R + v_L}{2} \qquad \omega = \frac{v_R - v_L}{L}$$

donde $L$ es la distancia entre las dos ruedas (**track width** o separación del eje).

La pose del robot en el plano ($x$, $y$, $\theta$) evoluciona según:

$$\dot{x} = v\cos\theta \qquad \dot{y} = v\sin\theta \qquad \dot{\theta} = \omega$$

### 1.3 Planificación de movimientos abiertos (open-loop)

Sin odometría ni sensores de retroalimentación, podemos calcular cuánto tiempo aplicar un comando para lograr un desplazamiento deseado:

$$t_{\text{lineal}} = \frac{d}{v} \qquad t_{\text{giro}} = \frac{|\Delta\theta|}{|\omega|}$$

Por ejemplo, para avanzar **0.5 m** a $v = 0.2$ m/s:
$$t = 0.5 / 0.2 = 2.5 \text{ s}$$

Para girar **90°** ($\pi/2$ rad) a $\omega = 0.5$ rad/s:
$$t = (\pi/2) / 0.5 \approx 3.14 \text{ s}$$

### 1.4 Radio de curvatura

Cuando $v \neq 0$ y $\omega \neq 0$ simultáneamente, el robot sigue un arco circular de radio:

$$R = \frac{v}{\omega}$$

Un radio grande → curva suave. Un radio pequeño → curva cerrada.

---

## 2. TurtleBot3 Burger

### 2.1 ¿Qué es el TurtleBot3?

El **TurtleBot3** es una plataforma robótica educativa y de investigación de código abierto desarrollada por **ROBOTIS** en colaboración con **Open Robotics**. Es uno de los robots de referencia oficial del ecosistema ROS, diseñado para ser asequible, personalizable y fácil de programar.

Existe en tres versiones: **Burger** (la más pequeña y ligera), **Waffle** y **Waffle Pi**.

### 2.2 Arquitectura del TurtleBot3 Burger

```
┌─────────────────────────────────────────────────┐
│              Computadora principal               │
│           Raspberry Pi 3 Model B+               │
│        (ROS2 Humble, Ubuntu 22.04)               │
└──────────────────┬──────────────────────────────┘
                   │ USB / Serial
┌──────────────────▼──────────────────────────────┐
│           Placa de control OpenCR               │
│      (STM32F746, firmware de bajo nivel)         │
│   • Control PID de motores                       │
│   • Lectura de encoders                          │
│   • Publicación de odometría                     │
│   • IMU integrada (MPU-9250)                     │
└──────┬──────────────────────────┬───────────────┘
       │ Protocolo Dynamixel       │
┌──────▼──────┐           ┌───────▼──────┐
│ Motor IZQ   │           │ Motor DER    │
│DYNAMIXEL    │           │DYNAMIXEL     │
│XL430-W250-T │           │XL430-W250-T  │
└─────────────┘           └──────────────┘
```

### 2.3 Motores — DYNAMIXEL XL430-W250-T

Los servomotores **DYNAMIXEL XL430-W250-T** son actuadores inteligentes con comunicación serial (protocolo Dynamixel v2). Sus características principales:

| Parámetro | Valor |
|-----------|-------|
| Voltaje de operación | 6.0 – 8.4 V |
| Torque máximo | 1.0 N·m (a 8.4 V) |
| Velocidad máxima | ~57 RPM (sin carga) |
| Resolución del encoder | 4096 pulsos/revolución (12 bits) |
| Reducción interna | 258.5 : 1 |
| Protocolo de comunicación | Dynamixel Protocol 2.0 (half-duplex TTL) |
| Retroalimentación | Posición, velocidad, corriente, temperatura |

Cada motor incluye **encoder absoluto** integrado, lo que permite medir rotaciones acumuladas y calcular odometría.

**Parámetros geométricos del Burger:**

| Parámetro | Valor |
|-----------|-------|
| Diámetro de rueda | 66 mm |
| Separación entre ruedas ($L$) | 160 mm |
| Velocidad lineal máxima | 0.22 m/s |
| Velocidad angular máxima | 2.84 rad/s |

### 2.4 Sensores

#### 2.4.1 LiDAR — LDS-01 (o RPLidar A1 como reemplazo)

El sensor principal de percepción del TurtleBot3 Burger es el **LiDAR de 360°**. Funciona emitiendo pulsos de luz infrarroja y midiendo el tiempo de vuelo (*Time of Flight*) de cada pulso para calcular distancias.

| Parámetro | LDS-01 | RPLidar A1 |
|-----------|--------|------------|
| Rango de escaneo | 360° | 360° |
| Frecuencia de muestreo | ~1800 muestras/s | 8000 muestras/s |
| Distancia mínima | 0.12 m | 0.15 m |
| Distancia máxima | 3.5 m | 12 m |
| Frecuencia de escaneo | ~5 Hz | 5.5 – 10 Hz |
| Interfaz | USB (CP2102) | USB (CP2102) |

El LiDAR publica en el tópico `/scan` con mensajes de tipo `sensor_msgs/LaserScan`.

#### 2.4.2 IMU — MPU-9250 (en OpenCR)

La placa OpenCR integra la **IMU MPU-9250**, un sensor de 9 grados de libertad (DoF):
- **Acelerómetro de 3 ejes**: mide aceleración lineal [m/s²]
- **Giroscopio de 3 ejes**: mide velocidad angular [rad/s]
- **Magnetómetro de 3 ejes**: mide campo magnético (orientación absoluta)

Publica en `/imu` con mensajes `sensor_msgs/Imu`.

#### 2.4.3 Encoders de rueda

Los encoders de los DYNAMIXEL permiten calcular la odometría del robot (estimación de posición y orientación a partir del conteo de pulsos). La OpenCR procesa esta información y publica en `/odom` con mensajes `nav_msgs/Odometry`.

### 2.5 Dimensiones y peso

| Parámetro | Valor |
|-----------|-------|
| Dimensiones (L×W×H) | 138 × 178 × 192 mm |
| Peso | ~1 kg |
| Batería | LiPo 11.1V 1800 mAh |
| Autonomía | ~2.5 h |

---

## 3. ROS2 — Robot Operating System 2

### 3.1 ¿Qué es ROS2?

**ROS2** (*Robot Operating System 2*) es un **framework de software de código abierto** para el desarrollo de sistemas robóticos. A pesar de su nombre, no es un sistema operativo — es una capa de middleware que corre sobre Linux (Ubuntu), macOS o Windows y proporciona:

- **Comunicación entre procesos** (nodos) de forma distribuida.
- **Herramientas de desarrollo**: visualización, logging, grabación y reproducción de datos.
- **Bibliotecas de algoritmos**: navegación, percepción, manipulación, etc.
- **Abstracción de hardware**: drivers para sensores, actuadores y plataformas robóticas.

### 3.2 Conceptos fundamentales

#### Nodos (*Nodes*)
Un **nodo** es un proceso independiente que realiza una tarea específica (leer un sensor, controlar un motor, procesar una imagen). Los nodos se comunican entre sí.

#### Tópicos (*Topics*) — comunicación publish/subscribe
Un **tópico** es un canal de comunicación con nombre. Un nodo **publica** mensajes en un tópico y otros nodos **se suscriben** para recibirlos. Es comunicación **asíncrona** y **uno-a-muchos**.

```
  [teleop_keyboard]  ──── /cmd_vel (Twist) ────►  [turtlebot3_node]
  [turtlebot3_node]  ──── /scan (LaserScan) ───►  [cartographer]
  [turtlebot3_node]  ──── /odom (Odometry) ────►  [rviz2]
```

#### Servicios (*Services*)
Comunicación **síncrona** de tipo petición/respuesta. Un nodo llama a un servicio y espera la respuesta. Útil para operaciones puntuales (guardar mapa, reiniciar sensor).

#### Acciones (*Actions*)
Similar a los servicios pero para tareas **de larga duración** con retroalimentación periódica. Ejemplo: "navega al punto (2, 3)" → el robot va reportando su progreso.

#### Mensajes (*Messages*)
Las estructuras de datos que viajan por tópicos. Ejemplos:
- `geometry_msgs/Twist` — velocidad lineal y angular
- `sensor_msgs/LaserScan` — escaneo LiDAR
- `nav_msgs/Odometry` — pose y velocidad estimadas

### 3.3 ¿Por qué ROS2 y no ROS1?

ROS2 fue rediseñado desde cero para superar las limitaciones de ROS1:

| Aspecto | ROS1 | ROS2 |
|---------|------|------|
| Middleware | XMLRPC custom (no estándar) | DDS estándar (FastDDS, CycloneDDS) |
| Nodo maestro | Requiere `roscore` (punto único de fallo) | Sin master centralizado |
| Tiempo real | No soportado | Soporte para sistemas en tiempo real |
| Seguridad | Sin cifrado ni autenticación | SROS2 (cifrado, autenticación, autorización) |
| Multi-robot | Difícil (networking manual) | Nativo mediante `ROS_DOMAIN_ID` |
| SO soportados | Solo Linux | Linux, macOS, Windows |
| Soporte activo | **EOL 2025** | Mantenido activamente |

### 3.4 ¿Por qué usar ROS2 en robótica?

1. **Reutilización**: miles de paquetes disponibles (navegación, SLAM, percepción, manipulación). No reinventas la rueda.
2. **Modularidad**: cada componente es un nodo independiente — fácil de depurar, reemplazar o escalar.
3. **Ecosistema**: es el estándar de facto en robótica académica e industrial.
4. **Simulación**: integración nativa con Gazebo para probar código antes de tocarlo en hardware real.
5. **Herramientas**: `rviz2` (visualización 3D), `ros2 bag` (grabación), `rqt` (GUI de debugging).

### 3.5 Comandos esenciales de ROS2

```bash
# Ver nodos activos
ros2 node list

# Ver tópicos activos
ros2 topic list

# Leer mensajes de un tópico en tiempo real
ros2 topic echo /scan

# Ver frecuencia de publicación
ros2 topic hz /cmd_vel

# Publicar manualmente en un tópico
ros2 topic pub /cmd_vel geometry_msgs/Twist "{linear: {x: 0.2}, angular: {z: 0.0}}"

# Ver información de un nodo
ros2 node info /turtlebot3_node

# Lanzar un archivo launch
ros2 launch turtlebot3_gazebo turtlebot3_world.launch.py
```

---

## 4. Gazebo — Simulador de Robótica

### 4.1 ¿Qué es Gazebo?

**Gazebo** es un simulador de robótica 3D de código abierto desarrollado por **Open Robotics**. Permite simular robots y sus entornos con física realista sin necesidad de hardware real.

Integra tres motores físicos principales:
- **ODE** (Open Dynamics Engine) — predeterminado
- **Bullet Physics**
- **DART**

### 4.2 ¿Qué simula Gazebo?

- **Física**: gravedad, colisiones, fricción, inercia, joints.
- **Sensores**: LiDAR, cámaras, IMU, GPS, ultrasonido → genera datos realistas con ruido configurable.
- **Actuadores**: motores, ruedas, brazos robóticos.
- **Entornos**: mundos 3D con obstáculos, paredes, iluminación.

### 4.3 ¿Por qué simular antes de usar hardware real?

| Ventaja | Descripción |
|---------|-------------|
| **Sin riesgo físico** | Un bug no rompe el robot ni lesiona personas |
| **Reproducibilidad** | El mismo experimento siempre en las mismas condiciones |
| **Velocidad** | Iterar código en segundos, no minutos de setup físico |
| **Escalabilidad** | Simular flotas de robots sin hardware |
| **Accesibilidad** | No se necesita el robot físico — basta una PC |

### 4.4 Integración Gazebo + ROS2

Gazebo se comunica con ROS2 a través del paquete `gazebo_ros_pkgs`. Una vez lanzado, el robot simulado publica y suscribe a los **mismos tópicos** que el robot real:

- Publica `/scan`, `/odom`, `/imu` como si fueran sensores reales.
- Suscribe a `/cmd_vel` para mover el robot.

Esto significa que **el mismo código funciona en simulación y en hardware real** sin modificaciones.

### 4.5 Lanzar el TurtleBot3 en Gazebo

```bash
# Configurar el modelo (en ~/.bashrc o antes de lanzar)
export TURTLEBOT3_MODEL=burger

# Lanzar mundo vacío
ros2 launch turtlebot3_gazebo empty_world.launch.py

# Lanzar mundo con obstáculos
ros2 launch turtlebot3_gazebo turtlebot3_world.launch.py

# Lanzar casa simulada
ros2 launch turtlebot3_gazebo turtlebot3_house.launch.py
```

---

## 5. Mapeo del Entorno Simulado

### 5.1 ¿Qué es SLAM?

**SLAM** (*Simultaneous Localization And Mapping*) es el problema de construir un mapa de un entorno desconocido mientras al mismo tiempo se estima la posición del robot dentro de ese mapa. Es el equivalente robótico a "explorar un lugar nuevo y recordar dónde has estado".

ROS2 incluye el paquete **Cartographer** de Google, que implementa SLAM 2D usando el LiDAR.

### 5.2 Procedimiento de mapeo en simulación

Abre **4 terminales** en TheConstruct y ejecuta en orden:

**Terminal 1 — Lanzar Gazebo con el TurtleBot3:**
```bash
export TURTLEBOT3_MODEL=burger
ros2 launch turtlebot3_gazebo turtlebot3_world.launch.py
```

**Terminal 2 — Lanzar Cartographer (SLAM):**
```bash
export TURTLEBOT3_MODEL=burger
ros2 launch turtlebot3_cartographer cartographer.launch.py use_sim_time:=True
```

**Terminal 3 — Controlar el robot con teclado:**
```bash
export TURTLEBOT3_MODEL=burger
ros2 run turtlebot3_teleop teleop_keyboard
```
> Usa las teclas `W/A/S/D` para mover el robot. Explora todo el entorno para construir el mapa completo.

**Terminal 4 — Guardar el mapa cuando esté listo:**
```bash
ros2 run nav2_map_server map_saver_cli -f ~/mi_mapa
```
Esto genera dos archivos: `mi_mapa.pgm` (imagen del mapa) y `mi_mapa.yaml` (metadatos).

### 5.3 Interpretación del mapa generado

El mapa resultante es una imagen en escala de grises donde:
- **Negro** → obstáculos (paredes, objetos)
- **Blanco** → espacio libre
- **Gris** → zona no explorada o incierta

> **Entrega requerida:** Incluye una captura de pantalla del mapa generado en tu reporte.

---

## 6. Tarea — Trayectoria Cerrada Personalizada

### 6.1 Descripción de la tarea

Deberás implementar un nodo ROS2 que haga que el TurtleBot3 ejecute una **trayectoria cerrada** (el robot termina en la misma posición y orientación aproximada donde empezó) que cumpla con los siguientes requisitos:

| Requisito | Descripción |
|-----------|-------------|
| **Figura cerrada** | El robot regresa al punto de inicio |
| **Mínimo 1 recta** | Al menos un segmento en línea recta |
| **Mínimo 1 diagonal** | Al menos un segmento en dirección diagonal (no a 0° ni 90°) |
| **Mínimo 1 curva** | Al menos un arco circular (v ≠ 0 y ω ≠ 0 simultáneamente) |
| **Complejidad** | Mínimo 5 segmentos de movimiento distintos |

**Ejemplos de figuras válidas:** trapecio redondeado, letra D, hoja de trébol simplificada, pentágono con una esquina curvada, etc.

### 6.2 Cómo funciona la plantilla

La función clave es `move(linear_x, angular_z, duration_secs)`:
- `linear_x` [m/s]: velocidad hacia adelante (negativo = atrás)
- `angular_z` [rad/s]: velocidad de giro (positivo = izquierda, negativo = derecha)
- `duration_secs` [s]: cuántos segundos mantener ese comando

**Recuerda:**
- Línea recta: `angular_z = 0`
- Giro en el lugar: `linear_x = 0`, `angular_z ≠ 0`
- Arco/curva: ambos `linear_x ≠ 0` y `angular_z ≠ 0`
- Para girar un ángulo θ (en radianes) a velocidad ω: `t = θ / ω`
- Para avanzar distancia d a velocidad v: `t = d / v`

In [ ]:
import math

# Constantes de referencia del TurtleBot3 Burger
MAX_LINEAR  = 0.22   # m/s
MAX_ANGULAR = 2.84   # rad/s

# Velocidades recomendadas para open-loop (más precisas a menor velocidad)
V_LINEAR  = 0.15   # m/s  — ajusta según tu figura
V_ANGULAR = 0.5    # rad/s — ajusta según tu figura

# ─────────────────────────────────────────────
#   FUNCIONES DE CÁLCULO DE TIEMPO
# ─────────────────────────────────────────────

def tiempo_lineal(distancia_m, velocidad=V_LINEAR):
    """Calcula el tiempo para recorrer una distancia en línea recta."""
    return distancia_m / velocidad

def tiempo_giro(angulo_deg, velocidad_angular=V_ANGULAR):
    """Calcula el tiempo para girar un ángulo dado en grados."""
    angulo_rad = math.radians(angulo_deg)
    return abs(angulo_rad) / velocidad_angular

def tiempo_arco(angulo_deg, radio_m, velocidad_angular=V_ANGULAR):
    """Calcula el tiempo y la velocidad lineal necesarios para un arco."""
    angulo_rad = math.radians(angulo_deg)
    t = abs(angulo_rad) / velocidad_angular
    v = radio_m * velocidad_angular
    return t, v

# Ejemplos de cálculo
print("=== Ejemplos de cálculo ===")
print(f"Avanzar 0.5 m a {V_LINEAR} m/s  → {tiempo_lineal(0.5):.2f} s")
print(f"Girar 90° a {V_ANGULAR} rad/s   → {tiempo_giro(90):.2f} s")
print(f"Girar 45° a {V_ANGULAR} rad/s   → {tiempo_giro(45):.2f} s")
print(f"Girar 180° a {V_ANGULAR} rad/s  → {tiempo_giro(180):.2f} s")
t_arco, v_arco = tiempo_arco(90, radio_m=0.3)
print(f"Arco 90°, R=0.3 m → t={t_arco:.2f} s, v_lineal={v_arco:.3f} m/s")

In [ ]:
#!/usr/bin/env python3
"""
Plantilla para trayectoria cerrada personalizada.
Completa las secciones marcadas con TODO.
"""
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist
import time
import math


class MiTrayectoria(Node):

    def __init__(self):
        super().__init__('mi_trayectoria')
        self.publisher = self.create_publisher(Twist, '/cmd_vel', 10)
        # Espera a que el robot y la simulación estén listos
        time.sleep(1.0)

    # ─────────────────────────────────────────────
    #   FUNCIÓN BASE — NO MODIFICAR
    # ─────────────────────────────────────────────
    def move(self, linear_x: float, angular_z: float, duration_secs: float):
        """Envía un comando de velocidad durante 'duration_secs' segundos."""
        msg = Twist()
        msg.linear.x  = float(linear_x)
        msg.angular.z = float(angular_z)

        end_time = time.time() + duration_secs
        while time.time() < end_time:
            self.publisher.publish(msg)
            time.sleep(0.05)  # 20 Hz

        self.stop()
        time.sleep(0.2)  # pequeña pausa entre movimientos

    def stop(self):
        self.publisher.publish(Twist())

    # ─────────────────────────────────────────────
    #   FUNCIONES DE AYUDA
    # ─────────────────────────────────────────────
    def avanzar(self, distancia_m, velocidad=0.15):
        """Avanza 'distancia_m' metros en línea recta."""
        t = distancia_m / velocidad
        self.move(velocidad, 0.0, t)

    def girar(self, angulo_deg, velocidad_angular=0.5):
        """
        Gira 'angulo_deg' grados en el lugar.
        Positivo = izquierda (CCW), negativo = derecha (CW).
        """
        angulo_rad = math.radians(angulo_deg)
        t = abs(angulo_rad) / velocidad_angular
        signo = 1.0 if angulo_deg >= 0 else -1.0
        self.move(0.0, signo * velocidad_angular, t)

    def arco(self, angulo_deg, radio_m, velocidad_angular=0.5):
        """
        Sigue un arco circular de 'radio_m' metros durante 'angulo_deg' grados.
        Positivo = curva a la izquierda, negativo = curva a la derecha.
        """
        angulo_rad = math.radians(angulo_deg)
        t = abs(angulo_rad) / velocidad_angular
        v_lineal = radio_m * velocidad_angular
        signo = 1.0 if angulo_deg >= 0 else -1.0
        self.move(v_lineal, signo * velocidad_angular, t)

    # ─────────────────────────────────────────────
    #   TODO: IMPLEMENTA TU TRAYECTORIA AQUÍ
    # ─────────────────────────────────────────────
    def ejecutar_trayectoria(self):
        """
        Define aquí tu trayectoria cerrada.

        REQUISITOS:
          - Al menos 1 recta       → usa self.avanzar(distancia)
          - Al menos 1 diagonal    → avanza en línea recta tras girar un ángulo no múltiplo de 90°
          - Al menos 1 curva/arco  → usa self.arco(angulo_deg, radio_m)
          - La figura debe ser CERRADA (regresar al punto de inicio)
          - Mínimo 5 segmentos de movimiento

        EJEMPLO — Triángulo rectángulo con una esquina redondeada
        (descomenta y modifica para entender la lógica, luego diseña tu propia figura):

            self.avanzar(0.5)           # Segmento 1: recta de 0.5 m
            self.girar(135)             # Giro de 135° (hacia diagonal)
            self.avanzar(0.35)          # Segmento 2: diagonal de 0.35 m
            self.girar(135)             # Giro hacia el inicio
            self.arco(90, 0.2)          # Segmento 3: arco 90°, radio 0.2 m
            self.girar(-90)             # Corrección de orientación
            self.avanzar(0.3)           # Segmento 4: recta final
            self.girar(90)              # Orientación original
        """

        self.get_logger().info('Iniciando trayectoria...')

        # ── SEGMENTO 1 ────────────────────────────────
        # TODO: describe qué hace este segmento
        # self.avanzar( ??? )

        # ── SEGMENTO 2 ────────────────────────────────
        # TODO: describe qué hace este segmento
        # self.girar( ??? )

        # ── SEGMENTO 3 ────────────────────────────────
        # TODO: describe qué hace este segmento (debe incluir una curva)
        # self.arco( ??? , ??? )

        # ── SEGMENTO 4 ────────────────────────────────
        # TODO: segmento diagonal

        # ── SEGMENTO 5 ────────────────────────────────
        # TODO: regreso al punto de inicio

        # Agrega más segmentos si tu figura lo requiere...

        self.get_logger().info('Trayectoria completada.')
        self.stop()


def main(args=None):
    rclpy.init(args=args)
    node = MiTrayectoria()
    try:
        node.ejecutar_trayectoria()
    except KeyboardInterrupt:
        pass
    finally:
        node.stop()
        node.destroy_node()
        rclpy.shutdown()


if __name__ == '__main__':
    main()

### 6.3 Cómo ejecutar tu trayectoria en Gazebo

Una vez que hayas completado la función `ejecutar_trayectoria()`, guarda el archivo y ejecútalo:

**Terminal 1 — Gazebo corriendo** (debe estar activo desde el paso de mapeo):
```bash
export TURTLEBOT3_MODEL=burger
ros2 launch turtlebot3_gazebo turtlebot3_world.launch.py
```

**Terminal 2 — Ejecutar tu script:**
```bash
python3 mi_trayectoria.py
```

> **Tip:** Si el robot no regresa exactamente al punto de inicio, es normal. Los errores de open-loop se acumulan. Ajusta los tiempos y velocidades iterativamente.

---

## 7. Entregables y Criterios de Evaluación

### 7.1 Entregables

1. **Captura del mapa generado** — imagen del entorno mapeado con Cartographer.
2. **Código de la trayectoria** — el archivo Python con tu implementación completa.
3. **Diagrama de la figura** — boceto de la figura geométrica que decidiste implementar, con medidas en metros y ángulos.
4. **Video o capturas de pantalla** — evidencia de la trayectoria ejecutándose en Gazebo.
5. **Reporte breve** — responde las siguientes preguntas:
   - ¿Cuál fue tu figura elegida y por qué?
   - ¿Cómo calculaste los tiempos de cada segmento?
   - ¿Qué diferencias observaste entre tu figura planeada y la ejecutada realmente? ¿A qué se deben?

### 7.2 Criterios de evaluación

| Criterio | Puntos |
|----------|--------|
| Mapa generado correctamente | 20 |
| Código funcional y bien estructurado | 25 |
| Figura cumple todos los requisitos (recta, diagonal, curva, cerrada) | 30 |
| Precisión del cierre de la trayectoria | 15 |
| Reporte de análisis | 10 |
| **Total** | **100** |

---

## 8. Referencias

- [TurtleBot3 e-Manual (ROBOTIS)](https://emanual.robotis.com/docs/en/platform/turtlebot3/overview/)
- [ROS2 Humble Documentation](https://docs.ros.org/en/humble/index.html)
- [Gazebo Classic Documentation](https://classic.gazebosim.org/tutorials)
- [DYNAMIXEL XL430-W250-T Datasheet](https://emanual.robotis.com/docs/en/dxl/x/xl430-w250/)
- Siegwart, R., Nourbakhsh, I.R., Scaramuzza, D. — *Introduction to Autonomous Mobile Robots*, 2nd ed., MIT Press, 2011.
- Correll, N. — *Introduction to Autonomous Robots*, GitHub Open Access, 2022.